Sistema de detección de enlaces spam

Queremos implementar un sistema que sea capaz de detectar automáticamente si una página web contiene spam o no basándonos en su URL.

Paso 1: Carga del conjunto de datos

In [50]:
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt
import seaborn as sns


from nltk import download
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.svm import SVC

import warnings
warnings.filterwarnings("ignore")
url="https://breathecode.herokuapp.com/asset/internal-link?id=932&path=url_spam.csv"

df=pd.read_csv(url)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2999 entries, 0 to 2998
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   url      2999 non-null   str  
 1   is_spam  2999 non-null   bool 
dtypes: bool(1), str(1)
memory usage: 26.5 KB


Paso 2: Preprocesa los enlaces

Transforma la variable booleana is_spamm en números (0 y 1)

In [51]:
df["is_spam"] = df["is_spam"].apply(lambda x: 1 if x == "spam" else 0).astype(int)

df.head()

,url,is_spam
0,https://briefingday.us8.list-manage.com/unsubs...,0
1,https://www.hvper.com/,0
2,https://briefingday.com/m/v4n3i4f3,0
3,https://briefingday.com/n/20200618/m#commentform,0
4,https://briefingday.com/fan,0


Eliminar duplicados

In [52]:
duplicados = df.duplicated()
num_duplicados = duplicados.sum()

print(f"El dataset tiene {num_duplicados} filas duplicadas")

El dataset tiene 630 filas duplicadas


In [53]:
df= df.drop_duplicates()
df= df.reset_index(inplace = False, drop = True)
df.shape

(2369, 2)

Procesamiento del texto

Función para todo a minúscula y eliminar signos y caracteres especiales

In [54]:
def preprocess_text(text):
    # Eliminar cualquier caracter que no sea una letra (a-z) o un espacio en blanco ( )
    text = re.sub(r'[^a-z ]', " ", text)

    # Eliminar espacios en blanco
    text = re.sub(r'\s+[a-zA-Z]\s+', " ", text)
    text = re.sub(r'\^[a-zA-Z]\s+', " ", text)

    # Reducir espacios en blanco múltiples a uno único
    text = re.sub(r'\s+', " ", text.lower())

    # Eliminar tags
    text = re.sub("&lt;/?.*?&gt;"," &lt;&gt; ", text)

    return text.split()

In [55]:
#aplicar los url a la función de procesaar texto
df["url"] = df["url"].apply(preprocess_text)
df.head()

,url,is_spam
0,"[https, briefingday, us, list, manage, com, un...",0
1,"[https, www, hvper, com]",0
2,"[https, briefingday, com, v, i]",0
3,"[https, briefingday, com, m, commentform]",0
4,"[https, briefingday, com, fan]",0


Lematización de texto

Función para lematización de texto con libreria nltk

In [56]:
# instancia lematizador
download("wordnet")
lemmatizer = WordNetLemmatizer()

download("stopwords")
stop_words = stopwords.words("english")

def lemmatize_text(words, lemmatizer = lemmatizer):
    # lematiza
    tokens = [lemmatizer.lemmatize(word) for word in words]
    # saca stop words
    tokens = [word for word in tokens if word not in stop_words]
    # se queda con las de largo mayor a
    tokens = [word for word in tokens if len(word) > 3]
    return tokens

[nltk_data] Downloading package wordnet to /home/vscode/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [57]:
#aplicar los url a la función de lematización
df["url"] = df["url"].apply(lemmatize_text)
df.head()

,url,is_spam
0,"[http, briefingday, list, manage, unsubscribe]",0
1,"[http, hvper]",0
2,"[http, briefingday]",0
3,"[http, briefingday, commentform]",0
4,"[http, briefingday]",0


Convertir en números

In [58]:
tokens_list = df["url"]
tokens_list = [" ".join(tokens) for tokens in tokens_list]

vectorizer = TfidfVectorizer(max_features = 5000, max_df = 0.8, min_df = 5)
X = vectorizer.fit_transform(tokens_list).toarray()
y = df["is_spam"]

X[:5]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(5, 538))

Paso 3: Construye un SVM

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [62]:
model = SVC(kernel = "linear", random_state = 42)

model.fit(X_train, y_train)

ValueError: The number of classes has to be greater than one; got 1 class